In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def Read_Email(emailId:str)->str:
    """This Tool Is For Reading Email Id Of User"""
    return f"Read EMail of {emailId}"


def SendEmail(recipent:str , Subject:str):
    """This Tool Is User To Send An Email"""
    return f"Send Email to {recipent} and subject is {Subject}"


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

models = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

agent = create_agent(
    model=models ,
    tools=[Read_Email , SendEmail],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
          interrupt_on={
              "SendEmail":{
                  "allowed_decisions":["approve" , "reject"],
              },
              "Read_Email":False,
          }
    )
    ]
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [25]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "Atifkhan"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send Email to test@example.com with subject Hello and body How are you?"
            )
        ]
    },
    config=config
)

print(response['messages'][-1].content)

[]


In [24]:
from langgraph.types import Command

if "__interrupt__" in response:
    print("Paused Approving................")

    response = agent.invoke(
        Command(
            resume={
                "decisions":[
                   { "type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(response['messages'][-1].content)
else:
    print('No interrupt found')

Paused Approving................
[{'type': 'text', 'text': 'I have sent the email to test@example.com with the subject "Hello". Please note that the current tool does not support adding a message body.'}]


In [ ]:
from langgraph.types import Command

if "__interrupt__" in response:
    print("Paused Approving................")

    response = agent.invoke(
        Command(
            resume={
                "decisions":[
                   { "type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(response['messages'][-1].content)
else:
    print('No interrupt found')

Paused Approving................
[{'type': 'text', 'text': "The email was not sent because the request was declined. Please let me know if you would like to try again or if there's anything else I can help you with."}]


: 